# 040 — Organise records to download / convert

Assembles the per-campaign ground-motion selection outputs from `033` (AvgSA([0, 3])) and `036`
(AvgSA([0, 6])) into the concrete work lists needed to fetch and prepare the records: NGA-Sub
download batchfiles, combined ESM/NGA-Sub download lists, and the AT2/HDF5→JSON conversion lists.
Runs in seconds; safe to re-run.

**Provenance guard** — before building anything it calls `cache_utils.verify(...)` on each upstream
selection CSV against its campaign's `final_ensembles.pickle`. A `StaleCacheError` means the CSVs
are stale and you must re-run `033` and/or `036` before continuing.

**Groups** — three groups are processed independently: `AvgSA_03`, `AvgSA_06`, and their `combined`
set. Overlap between the two campaigns is intentional and left in place (no cross-group de-dup).

**Upstream** (from `033`/`036`, in `cfg["proc_data"]["gm_selection"]`; each carries a
`.manifest.json` provenance sidecar):
- `{campaign}_summary_esm_records_selected.csv`
- `{campaign}_summary_ngasub_records_selected.csv`
- `{campaign}_records_to_convert.csv`

Also reads already-downloaded state from `cfg["raw_data"]["esm_hdf5_folder"]` and
`cfg["raw_data"]["ngasub_folder"]` to skip records already on disk.

**Output** (in `cfg["proc_data"]["gm_selection"]`; each stamped with a `.manifest.json`):
- `ngasub_rsns_to_download_batched_{combined,AvgSA_06,AvgSA_03}.csv` — per-group NGA-Sub batchfiles
  (30 RSNs/row, 1-indexed row number, no header), uploaded by hand to the NGA-Sub download site.
- `all_esm_records_to_download.csv`, `all_ngasub_records_to_download.csv` — combined download lists
  (selected records still missing locally).
- `esm_records_to_convert.csv`, `ngasub_records_to_convert.csv` — combined AT2/HDF5→JSON conversion
  lists.

**Run order** — run top to bottom after `033` and `036` have (re)written their selection CSVs.

In [16]:
import pandas as pd

from phd_project.config import config


cfg = config.load_config()

## Provenance guard

Verify the upstream selection CSVs are fresh before building any lists.

In [17]:
from phd_project.scripts.cache_utils import fingerprint, verify, write_manifest

# Provenance guard: refuse to build download/convert lists from stale selection
# CSVs. Each CSV was stamped by nb 033/036 against its campaign's
# final_ensembles.pickle; verify that stamp still matches before consuming it.
# A StaleCacheError here means you must re-run nb 033 and/or 036.
gm_dir = cfg["proc_data"]["gm_selection"]
for campaign in ("AvgSA_03", "AvgSA_06"):
    src = fingerprint(final_ensembles=gm_dir / f"{campaign}_final_ensembles.pickle")
    for name in (f"{campaign}_summary_esm_records_selected.csv",
                 f"{campaign}_summary_ngasub_records_selected.csv",
                 f"{campaign}_records_to_convert.csv"):
        verify(gm_dir / name, src)
print("Upstream selection CSVs verified fresh.")

Upstream selection CSVs verified fresh.


## Build per-group download lists + NGA-Sub batchfiles

For each group (`combined`, `AvgSA_06`, `AvgSA_03`): combine the campaigns' selected records, drop those already downloaded, write a per-group NGA-Sub batchfile, and (for `combined`) the combined ESM/NGA-Sub download lists.

In [18]:
import csv

# --- groups to process (overlap between AvgSA_03 and AvgSA_06 is intentional) ---
groups = {
    "combined": ["AvgSA_06", "AvgSA_03"],
    "AvgSA_06": ["AvgSA_06"],
    "AvgSA_03": ["AvgSA_03"],
}

gm_dir = cfg["proc_data"]["gm_selection"]

# combined download-CSV paths (written for the "combined" group only, unchanged names)
esm_combined_fp = gm_dir / f"all_esm_records_to_download.csv"
ngasub_combined_fp = gm_dir / f"all_ngasub_records_to_download.csv"

# --- already-downloaded sets (computed once) ---
# ESM: filenames look like  {event_id}__{station_code}__{location_code}.h5
downloaded_esm = set()
for f in cfg["raw_data"]["esm_hdf5_folder"].glob("*.h5"):
    parts = f.stem.split("__")
    if len(parts) == 3:
        ev, st, loc = parts
        downloaded_esm.add((ev, st, int(loc)))  # normalise "00" -> 0

# NGAsub: subdirs named NGASub_RSN_<rsn>
downloaded_ngasub = {
    int(d.name.removeprefix("NGASub_RSN_"))
    for d in cfg["raw_data"]["ngasub_folder"].iterdir()
    if d.name.startswith("NGASub_RSN_")
}


def process_group(label, campaigns, write_combined_csvs=False):
    # --- ESM: combine the selected records from the group's campaigns ---
    esm_selected = pd.concat(
        [pd.read_csv(gm_dir / f"{c}_summary_esm_records_selected.csv", header=[0, 1])
         for c in campaigns],
        ignore_index=True,
    ).drop_duplicates()

    esm_keys = list(zip(
        esm_selected[("metadata", "event_id")],
        esm_selected[("metadata", "station_code")],
        esm_selected[("metadata", "location_code")].astype(int),
    ))
    esm_to_download = esm_selected[[k not in downloaded_esm for k in esm_keys]].reset_index(drop=True)

    # --- NGAsub: combine the selected records from the group's campaigns ---
    nga_selected = pd.concat(
        [pd.read_csv(gm_dir / f"{c}_summary_ngasub_records_selected.csv", header=0)
         for c in campaigns]
    )
    nga_selected = (nga_selected.groupby("NGAsubRSN")["count"].sum()
                    .reset_index().sort_values("count", ascending=False))
    nga_to_download = nga_selected[~nga_selected["NGAsubRSN"].isin(downloaded_ngasub)].reset_index(drop=True)

    # batched RSN file: 30 RSNs per row, first cell = 1-indexed row number, no header
    ngasub_rsn_batches_fp = gm_dir / f"{label}_ngasub_rsns_to_download_batched.csv"
    rsns = nga_to_download["NGAsubRSN"].tolist()
    rsn_rows = [[i // 30 + 1, *rsns[i:i + 30]] for i in range(0, len(rsns), 30)]
    with open(ngasub_rsn_batches_fp, "w", newline="") as fh:
        csv.writer(fh).writerows(rsn_rows)
    # provenance: this batchfile derives only from the group's ngasub summaries
    write_manifest(ngasub_rsn_batches_fp, fingerprint(**{
        f"{c}_ngasub_selected": gm_dir / f"{c}_summary_ngasub_records_selected.csv"
        for c in campaigns}))

    # combined download CSVs (written for the "combined" group only, unchanged names)
    if write_combined_csvs:
        esm_to_download.to_csv(esm_combined_fp, index=False)
        write_manifest(esm_combined_fp, fingerprint(**{
            f"{c}_esm_selected": gm_dir / f"{c}_summary_esm_records_selected.csv"
            for c in campaigns}))
        nga_to_download.to_csv(ngasub_combined_fp, index=False)
        write_manifest(ngasub_combined_fp, fingerprint(**{
            f"{c}_ngasub_selected": gm_dir / f"{c}_summary_ngasub_records_selected.csv"
            for c in campaigns}))

    # --- summary ---
    print(f"=== {label} ({' + '.join(campaigns)}) ===")
    print("Records still to download (selected minus already downloaded):")
    print(f"  ESM:    {len(esm_to_download):>5} / {len(esm_selected)} selected  "
          f"({len(esm_selected) - len(esm_to_download)} already downloaded)")
    print(f"  NGAsub: {len(nga_to_download):>5} / {len(nga_selected)} selected  "
          f"({len(nga_selected) - len(nga_to_download)} already downloaded)")
    print(f"  NGAsub batched RSN file: {ngasub_rsn_batches_fp.name}  "
          f"({len(rsn_rows)} rows of up to 30)")
    print()


for label, campaigns in groups.items():
    process_group(label, campaigns, write_combined_csvs=(label == "combined"))

=== combined (AvgSA_06 + AvgSA_03) ===
Records still to download (selected minus already downloaded):
  ESM:        0 / 1180 selected  (1180 already downloaded)
  NGAsub:   608 / 1458 selected  (850 already downloaded)
  NGAsub batched RSN file: combined_ngasub_rsns_to_download_batched.csv  (21 rows of up to 30)

=== AvgSA_06 (AvgSA_06) ===
Records still to download (selected minus already downloaded):
  ESM:        0 / 864 selected  (864 already downloaded)
  NGAsub:   464 / 977 selected  (513 already downloaded)
  NGAsub batched RSN file: AvgSA_06_ngasub_rsns_to_download_batched.csv  (16 rows of up to 30)

=== AvgSA_03 (AvgSA_03) ===
Records still to download (selected minus already downloaded):
  ESM:        0 / 641 selected  (641 already downloaded)
  NGAsub:   174 / 775 selected  (601 already downloaded)
  NGAsub batched RSN file: AvgSA_03_ngasub_rsns_to_download_batched.csv  (6 rows of up to 30)



## Records to convert

Combine the per-campaign conversion lists into the per-database AT2/HDF5→JSON conversion lists, and report the conversion counts per group.

In [19]:
# paths to per-campaign conversion lists:
convert_AvgSA03_fp = gm_dir / f"AvgSA_03_records_to_convert.csv"
convert_AvgSA06_fp = gm_dir / f"AvgSA_06_records_to_convert.csv"

esm_convert_fp = gm_dir / f"esm_records_to_convert.csv"
ngasub_convert_fp = gm_dir / f"ngasub_records_to_convert.csv"

conv_03 = pd.read_csv(convert_AvgSA03_fp, dtype=str)
conv_06 = pd.read_csv(convert_AvgSA06_fp, dtype=str)
conv_combined = pd.concat([conv_03, conv_06], ignore_index=True).drop_duplicates()

# provenance: the combined convert lists always derive from both campaigns' lists
convert_provenance = fingerprint(
    AvgSA_03_records_to_convert=convert_AvgSA03_fp,
    AvgSA_06_records_to_convert=convert_AvgSA06_fp,
)

# write the combined per-database conversion lists (unchanged behaviour)
for db_name, fp in [("ESM", esm_convert_fp), ("NGASub", ngasub_convert_fp)]:
    sub = (conv_combined[conv_combined["database"] == db_name]
           [["record_identifier", "component"]]
           .drop_duplicates()
           .reset_index(drop=True))
    sub.to_csv(fp, index=False)
    write_manifest(fp, convert_provenance)
    print(f"{fp.name}: {len(sub)} records")

# --- per-group conversion counts (reuses `groups` from cell two) ---
print()
print("Records to convert per group:")
for label, campaigns in groups.items():
    conv = pd.concat(
        [pd.read_csv(gm_dir / f"{c}_records_to_convert.csv", dtype=str) for c in campaigns],
        ignore_index=True,
    ).drop_duplicates()
    counts = {
        db: len(conv[conv["database"] == db][["record_identifier", "component"]].drop_duplicates())
        for db in ("ESM", "NGASub")
    }
    print(f"  {label:>8}: ESM {counts['ESM']:>5}   NGASub {counts['NGASub']:>5}")

esm_records_to_convert.csv: 1560 records
ngasub_records_to_convert.csv: 1710 records

Records to convert per group:
  combined: ESM  1560   NGASub  1710
  AvgSA_06: ESM  1078   NGASub  1102
  AvgSA_03: ESM   845   NGASub   863
